In [3]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

print("1. Loading datasets...")
base_df = pd.read_csv('global_bleaching_environmental.csv', low_memory=False)
plastic_df = pd.read_csv('ocean_plastic_pollution_data.csv')

1. Loading datasets...


In [4]:
print("2. Cleaning missing coordinates...")
base_valid = base_df.dropna(subset=['Latitude_Degrees', 'Longitude_Degrees']).copy()
plastic_valid = plastic_df.dropna(subset=['Latitude', 'Longitude']).copy()

print("3. Performing spatial merge (Finding nearest plastic hotspot)...")
base_radians = np.radians(base_valid[['Latitude_Degrees', 'Longitude_Degrees']].values)
plastic_radians = np.radians(plastic_valid[['Latitude', 'Longitude']].values)

# Build BallTree using Haversine distance
tree = BallTree(plastic_radians, metric='haversine')
distances, indices = tree.query(base_radians, k=1)

# Map features back (convert radians to kilometers)
base_valid['Distance_to_Plastic_km'] = distances.flatten() * 6371
base_valid['Nearest_Plastic_Weight_kg'] = plastic_valid.iloc[indices.flatten()]['Plastic_Weight_kg'].values
print("4. Formatting the Target Variable and cleaning predictive features...")
# The base paper predicted 'Percent_Bleaching'. We must convert it to a number.
base_valid['Percent_Bleaching'] = pd.to_numeric(base_valid['Percent_Bleaching'], errors='coerce')

# Keep only rows where bleaching actually happened and is recorded
model_df = base_valid.dropna(subset=['Percent_Bleaching'])

# Define our predictive features
features_to_keep = ['ClimSST', 'Temperature_Mean', 'Depth_m', 'Windspeed', 'SSTA']

# Ensure all features are numeric and drop rows with missing environmental data
for col in features_to_keep:
    model_df.loc[:, col] = pd.to_numeric(model_df[col], errors='coerce')
    
model_df = model_df.dropna(subset=features_to_keep)

2. Cleaning missing coordinates...
3. Performing spatial merge (Finding nearest plastic hotspot)...
4. Formatting the Target Variable and cleaning predictive features...


In [7]:
# Create the final, clean dataset with just what the models need
cols_to_keep = ['Latitude_Degrees', 'Longitude_Degrees'] + features_to_keep + \
               ['Distance_to_Plastic_km', 'Nearest_Plastic_Weight_kg', 'Percent_Bleaching']

final_df = model_df[cols_to_keep].copy()

# Save for Step 2
output_file = 'combined_corals.csv'
final_df.to_csv(output_file, index=False)

print(f"\nMerged dataset saved as: {output_file}")
print(f"Shape of ready dataset: {final_df.shape}")
print("\nPreview of final features:")
print(final_df.head(3).to_string())


Merged dataset saved as: combined_corals.csv
Shape of ready dataset: (32716, 10)

Preview of final features:
   Latitude_Degrees  Longitude_Degrees ClimSST Temperature_Mean Depth_m Windspeed  SSTA  Distance_to_Plastic_km  Nearest_Plastic_Weight_kg  Percent_Bleaching
0            23.163           -82.5260  301.61           300.67    10.0       8.0 -0.46               67.237512                     275.27               50.2
1           -17.575          -149.7833  262.15           300.73    14.0       2.0  1.29              247.719898                     357.79               50.7
2            18.369           -64.5640  298.79           300.32     7.0       8.0  0.04              210.875742                     366.17               50.9
